<a href="https://colab.research.google.com/github/ALX-UFG-AKCIT-GEN-AI-2026/tcc/blob/main/bpmn-workflow-generator/prototype/colab/Fluxo_BPM_Demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Conexão com o drive

In [ ]:
!pip install -q -U \
    transformers \
    datasets \
    peft \
    trl \
    accelerate \
    bitsandbytes

Fazendo o download do Modelo Qwen/Qwen2.5-Coder-7B-Instruct

In [ ]:
import os
import gdown
url = "https://huggingface.co/alexfdealmeida/qwen25-coder-7b-tcc/resolve/main/Qwen2.5-Coder-7B-Instruct.zip?download=true"
destino = "/content/Qwen2.5-Coder-7B-Instruct.zip"
gdown.download(url, destino, quiet=False)
print(f"Arquivo salvo em: {destino}")

Descompactando

In [ ]:
import zipfile
with zipfile.ZipFile('/content/Qwen2.5-Coder-7B-Instruct.zip', 'r') as zip_ref:
    zip_ref.extractall('/')

In [ ]:
import torch
import json
from transformers import (
    pipeline,
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)
modelo = "/content/Qwen2.5-Coder-7B-Instruct"

# configuração de quantização
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# tokenizer
tokenizer = AutoTokenizer.from_pretrained(modelo)

# modelo quantizado
model = AutoModelForCausalLM.from_pretrained(
    modelo,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)

# pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

In [ ]:
import requests
url = "https://github.com/ALX-UFG-AKCIT-GEN-AI-2026/tcc/raw/refs/heads/main/bpmn-workflow-generator/prototype/models/qwen-bpmn-lora-final.zip"
destino = "/content/qwen-bpmn-lora-final.zip"
resposta = requests.get(url, timeout=300)
resposta.raise_for_status()
with open(destino, "wb") as arquivo:
    arquivo.write(resposta.content)
print(f"Download concluído: {destino}")
print(f"Tamanho: {len(resposta.content) / (1024**2):.2f} MB")

Fazendo o download do lora fine tuning

Descompactando

In [ ]:
import shutil
shutil.unpack_archive(
    "/content/qwen-bpmn-lora-final.zip",
    "/content/qwen-bpmn-lora-final"
)
print("Descompactado!")

In [ ]:
print(model.config.num_hidden_layers)
print(model.config.hidden_size)

In [ ]:
print(torch.cuda.get_device_name(0))

Código Geração BPMN

In [ ]:
import json
import os
from lxml import etree

class MotorGeradorBPMN:
    def __init__(self, dados_formulario):
        self.metadata = dados_formulario.get("metadata", {})
        self.nome_processo = self.metadata.get("nome", "Processo Customizado")
        self.id_processo = self.metadata.get("id", "processo_customizado")
        self.lanes = self.metadata.get("lanes", ["Setor Padrão"])
        self.elementos = dados_formulario.get("elementos", {})

        # Namespaces oficiais para total compatibilidade com Camunda e Bizagi
        self.nsmap = {
            "bpmn": "http://www.omg.org/spec/BPMN/20100524/MODEL",
            "bpmndi": "http://www.omg.org/spec/BPMN/20100524/DI",
            "dc": "http://www.omg.org/spec/DD/20100524/DC",
            "di": "http://www.omg.org/spec/DD/20100524/DI"
        }

        # Raiz do documento XML
        self.root = etree.Element(f"{{{self.nsmap['bpmn']}}}definitions", nsmap=self.nsmap, id="Definitions_Form", targetNamespace="http://bpmn.io/schema/bpmn", exporter="MotorBPMN", exporterVersion="1.0")
        self.collaboration = etree.SubElement(self.root, f"{{{self.nsmap['bpmn']}}}collaboration", id="Collaboration_1")
        self.participant = etree.SubElement(self.collaboration, f"{{{self.nsmap['bpmn']}}}participant", id="Participant_1", name=self.nome_processo, processRef=self.id_processo)

        self.process = etree.SubElement(self.root, f"{{{self.nsmap['bpmn']}}}process", id=self.id_processo, isExecutable="true")
        self.lane_set = etree.SubElement(self.process, f"{{{self.nsmap['bpmn']}}}laneSet", id="LaneSet_1")

        self.diagram = etree.SubElement(self.root, f"{{{self.nsmap['bpmndi']}}}BPMNDiagram", id="BPMNDiagram_1")
        self.plane = etree.SubElement(self.diagram, f"{{{self.nsmap['bpmndi']}}}BPMNPlane", id="BPMNPlane_1", bpmnElement="Collaboration_1")

        self.coordenadas = {}
        self.contador_fluxos = 0
        self.altura_raia = 280
        self.largura_total_pool = 2000

    def calcular_posicoes_grid(self):
        """ Calcula o layout ortogonal distribuindo o X sequencialmente e fixando o Y por Lane """
        altura_raia = self.altura_raia
        posicoes_raias = {lane: (i * altura_raia) + 150 for i, lane in enumerate(self.lanes)}

        id_inicio = next((k for k, v in self.elementos.items() if v["tipo"] == "start"), None)
        if not id_inicio:
            id_inicio = list(self.elementos.keys())[0] if self.elementos else None

        if not id_inicio:
            return

        # Algoritmo de Varredura em Largura (BFS) para ordenação de dependências visuais
        fila = [(id_inicio, 220)]
        visitados = set()

        while fila:
            id_no, x_atual = fila.pop(0)
            if id_no in visitados or id_no not in self.elementos:
                continue

            no = self.elementos[id_no]
            lane_atual = no.get("lane", self.lanes[0])
            centro_y = posicoes_raias.get(lane_atual, 150) + 70

            self.coordenadas[id_no] = (x_atual, centro_y)
            visitados.add(id_no)

            proximos = no.get("proximos", [])
            for prox in proximos:
                prox_id = prox if isinstance(prox, str) else prox.get("destino")
                fila.append((prox_id, x_atual + 240))

    def compilar_xml(self):
        self.calcular_posicoes_grid()
        altura_raia = self.altura_raia
        largura_total_pool = self.largura_total_pool

        # Desenha a Pool Geral
        shape_part = etree.SubElement(self.plane, f"{{{self.nsmap['bpmndi']}}}BPMNShape", id="Participant_1_di", bpmnElement="Participant_1", isHorizontal="true")
        etree.SubElement(shape_part, f"{{{self.nsmap['dc']}}}Bounds", x="120", y="100", width=str(largura_total_pool), height=str(altura_raia * len(self.lanes)))

        # Desenha as divisórias das Raias (Lanes)
        elementos_raias = {}
        for i, nome_lane in enumerate(self.lanes):
            id_lane = f"Lane_{i}"
            el_lane = etree.SubElement(self.lane_set, f"{{{self.nsmap['bpmn']}}}lane", id=id_lane, name=nome_lane)
            elementos_raias[nome_lane] = el_lane

            shape_lane = etree.SubElement(self.plane, f"{{{self.nsmap['bpmndi']}}}BPMNShape", id=f"{id_lane}_di", bpmnElement=id_lane, isHorizontal="true")
            etree.SubElement(shape_lane, f"{{{self.nsmap['dc']}}}Bounds", x="150", y=str((i * altura_raia) + 100), width=str(largura_total_pool - 30), height=str(altura_raia))

        # Criação dos Objetos de Fluxo (Nodes)
        for id_no, no in self.elementos.items():
            tipo = no["tipo"]
            texto = no["texto"]
            nome_lane = no.get("lane", self.lanes[0])
            x, y = self.coordenadas.get(id_no, (220, 150))

            if nome_lane in elementos_raias:
                etree.SubElement(elementos_raias[nome_lane], f"{{{self.nsmap['bpmn']}}}flowNodeRef").text = id_no

            if tipo == "start":
                etree.SubElement(self.process, f"{{{self.nsmap['bpmn']}}}startEvent", id=id_no, name=texto)
                shape = etree.SubElement(self.plane, f"{{{self.nsmap['bpmndi']}}}BPMNShape", id=f"{id_no}_di", bpmnElement=id_no)
                etree.SubElement(shape, f"{{{self.nsmap['dc']}}}Bounds", x=str(x), y=str(y-18), width="36", height="36")

            elif tipo == "end":
                etree.SubElement(self.process, f"{{{self.nsmap['bpmn']}}}endEvent", id=id_no, name=texto)
                shape = etree.SubElement(self.plane, f"{{{self.nsmap['bpmndi']}}}BPMNShape", id=f"{id_no}_di", bpmnElement=id_no)
                etree.SubElement(shape, f"{{{self.nsmap['dc']}}}Bounds", x=str(x), y=str(y-18), width="36", height="36")

            elif tipo in ["gateway_split", "gateway_merge"]:
                etree.SubElement(self.process, f"{{{self.nsmap['bpmn']}}}exclusiveGateway", id=id_no, name=texto)
                shape = etree.SubElement(self.plane, f"{{{self.nsmap['bpmndi']}}}BPMNShape", id=f"{id_no}_di", bpmnElement=id_no)
                etree.SubElement(shape, f"{{{self.nsmap['dc']}}}Bounds", x=str(x), y=str(y-50), width="50", height="50")
            # criação do gateway_parallel
            elif tipo in ["gateway_parallel", "gateway_parallel_merge"]:
                etree.SubElement(self.process, f"{{{self.nsmap['bpmn']}}}parallelGateway", id=id_no, name=texto)
                shape = etree.SubElement(self.plane, f"{{{self.nsmap['bpmndi']}}}BPMNShape", id=f"{id_no}_di", bpmnElement=id_no)
                etree.SubElement(shape, f"{{{self.nsmap['dc']}}}Bounds", x=str(x), y=str(y-40), width="50", height="50")

            elif tipo in ["gateway_inclusivo", "gateway_inclusivo_merge"]:
                # Alterado para 'inclusiveGateway' (simbolizado por um círculo)
                etree.SubElement(self.process, f"{{{self.nsmap['bpmn']}}}inclusiveGateway", id=id_no, name=texto)
                shape = etree.SubElement(self.plane, f"{{{self.nsmap['bpmndi']}}}BPMNShape", id=f"{id_no}_di", bpmnElement=id_no)
                etree.SubElement(shape, f"{{{self.nsmap['dc']}}}Bounds", x=str(x), y=str(y-30), width="50", height="50")
            elif tipo in ["intermediate_event", "intermediate_event_merge"]:
                # Alterado para 'inclusiveGateway' (simbolizado por um círculo)
                etree.SubElement(self.process, f"{{{self.nsmap['bpmn']}}}intermediateCatchEvent", id=id_no, name=texto)
                shape = etree.SubElement(self.plane, f"{{{self.nsmap['bpmndi']}}}BPMNShape", id=f"{id_no}_di", bpmnElement=id_no)
                etree.SubElement(shape, f"{{{self.nsmap['dc']}}}Bounds", x=str(x), y=str(y-25), width="20", height="20")



            elif tipo == "tarefa":
                etree.SubElement(self.process, f"{{{self.nsmap['bpmn']}}}userTask", id=id_no, name=texto)
                shape = etree.SubElement(self.plane, f"{{{self.nsmap['bpmndi']}}}BPMNShape", id=f"{id_no}_di", bpmnElement=id_no)
                etree.SubElement(shape, f"{{{self.nsmap['dc']}}}Bounds", x=str(x), y=str(y-40), width="100", height="80")

            elif tipo == "subprocesso":
                etree.SubElement(self.process, f"{{{self.nsmap['bpmn']}}}subProcess", id=id_no, name=texto)
                shape = etree.SubElement(self.plane, f"{{{self.nsmap['bpmndi']}}}BPMNShape", id=f"{id_no}_di", bpmnElement=id_no)
                etree.SubElement(shape, f"{{{self.nsmap['dc']}}}Bounds", x=str(x), y=str(y-40), width="100", height="80")

        # Conexão de Setas e Fluxos (Sequence Flows)
        for id_no, no in self.elementos.items():
            for prox in no.get("proximos", []):
                id_prox = prox if isinstance(prox, str) else prox.get("destino")
                texto_cond = "" if isinstance(prox, str) else prox.get("condicao", "")

                if id_prox in self.coordenadas:
                    self.contador_fluxos += 1
                    id_fluxo = f"Flow_{self.contador_fluxos}"

                    sf = etree.SubElement(self.process, f"{{{self.nsmap['bpmn']}}}sequenceFlow", id=id_fluxo, sourceRef=id_no, targetRef=id_prox)
                    if texto_cond:
                        sf.set("name", texto_cond)

                    edge = etree.SubElement(self.plane, f"{{{self.nsmap['bpmndi']}}}BPMNEdge", id=f"{id_fluxo}_di", bpmnElement=id_fluxo)
                    x_orig, y_orig = self.coordenadas[id_no]
                    x_dest, y_dest = self.coordenadas[id_prox]

                    offset_orig = 100 if no["tipo"] == "tarefa" else (50 if "gateway" in no["tipo"] else 36)
                    etree.SubElement(edge, f"{{{self.nsmap['di']}}}waypoint", x=str(x_orig + offset_orig), y=str(y_orig))
                    etree.SubElement(edge, f"{{{self.nsmap['di']}}}waypoint", x=str(x_dest), y=str(y_dest))

    def salvar(self, nome_arquivo):
        self.compilar_xml()
        with open(nome_arquivo, "wb") as f:
            f.write(etree.tostring(self.root, pretty_print=True, xml_declaration=True, encoding="UTF-8"))
formularios_completos = {
  "concessao_diarias_passagens.bpmn": {
    "metadata": {
      "nome": "Concessão de Diárias e Passagens",
      "id": "proc_diarias_passagens",
      "lanes": [
        "Proponente / Solicitante",
        "Chefia Imediata",
        "Setor de Viagens / Protocolo",
        "Ordenador de Despesas"
      ]
    },
    "elementos": {
      "start_solicitacao": {
        "tipo": "start",
        "texto": "O Proponente preenche o formulário de solicitação de afastamento, informando o roteiro, justificativa e datas da viagem.",
        "lane": "Proponente / Solicitante",
        "proximos": [
          "analise_chefia"
        ]
      },
      "analise_chefia": {
        "tipo": "tarefa",
        "texto": "A Chefia Imediata analisa a solicitação do proponente.",
        "lane": "Chefia Imediata",
        "proximos": [
          "gateway_aprovacao_chefia"
        ]
      },
      "gateway_aprovacao_chefia": {
        "tipo": "gateway_split",
        "texto": "O pedido é aprovado?",
        "lane": "Chefia Imediata",
        "proximos": [
          {
            "destino": "parecer_desfavoravel",
            "condicao": "Não"
          },
          {
            "destino": "encaminhar_setor_viagens",
            "condicao": "Sim"
          }
        ]
      },
      "parecer_desfavoravel": {
        "tipo": "tarefa",
        "texto": "A chefia emite parecer desfavorável, o processo é arquivado e o fluxo é encerrado por indeferimento.",
        "lane": "Chefia Imediata",
        "proximos": [
          "end_indeferimento"
        ]
      },
      "end_indeferimento": {
        "tipo": "end",
        "texto": "Fluxo encerrado por indeferimento.",
        "lane": "Chefia Imediata",
        "proximos": []
      },
      "encaminhar_setor_viagens": {
        "tipo": "tarefa",
        "texto": "O processo é encaminhado ao Setor de Viagens.",
        "lane": "Chefia Imediata",
        "proximos": [
          "calcular_valores_viagem"
        ]
      },
      "calcular_valores_viagem": {
        "tipo": "tarefa",
        "texto": "O Setor de Viagens calcula os valores de diárias e realiza a cotação/emissão das passagens.",
        "lane": "Setor de Viagens / Protocolo",
        "proximos": [
          "encaminhar_ordenador"
        ]
      },
      "encaminhar_ordenador": {
        "tipo": "tarefa",
        "texto": "O processo é encaminhado ao Ordenador de Despesas para autorização do pagamento e empenho.",
        "lane": "Setor de Viagens / Protocolo",
        "proximos": [
          "gateway_aprovacao_ordenador"
        ]
      },
      "gateway_aprovacao_ordenador": {
        "tipo": "gateway_split",
        "texto": "O ordenador aprova a despesa?",
        "lane": "Ordenador de Despesas",
        "proximos": [
          {
            "destino": "retornar_setor_viagens",
            "condicao": "Não"
          },
          {
            "destino": "autorizar_pagamento",
            "condicao": "Sim"
          }
        ]
      },
      "retornar_setor_viagens": {
        "tipo": "tarefa",
        "texto": "Retorna ao Setor de Viagens para ajustes ou cancelamento.",
        "lane": "Ordenador de Despesas",
        "proximos": [
          "calcular_valores_viagem"
        ]
      },
      "autorizar_pagamento": {
        "tipo": "tarefa",
        "texto": "O pagamento é autorizado e o bilhete/ordem de pagamento é emitido.",
        "lane": "Ordenador de Despesas",
        "proximos": [
          "realizar_viagem"
        ]
      },
      "realizar_viagem": {
        "tipo": "tarefa",
        "texto": "O proponente realiza a viagem e, após o retorno, submete o relatório de viagem e os comprovantes de embarque.",
        "lane": "Proponente / Solicitante",
        "proximos": [
          "conferir_documentacao"
        ]
      },
      "conferir_documentacao": {
        "tipo": "tarefa",
        "texto": "O Setor de Viagens confere a documentação comprobatória e o processo é arquivado com sucesso.",
        "lane": "Setor de Viagens / Protocolo",
        "proximos": [
          "end_sucesso"
        ]
      },
      "end_sucesso": {
        "tipo": "end",
        "texto": "Processo arquivado e concluído com sucesso.",
        "lane": "Setor de Viagens / Protocolo",
        "proximos": []
      }
    }
  },
  "emitir_declaracao_sipac.bpmn": {
    "metadata": {
      "nome": "Emitir declaração no SIPAC",
      "id": "proc-emitir-declaracao-sipac",
      "lanes": [
        "Bolsista do RU",
        "DASAP"
      ]
    },
    "elementos": {
      "start_acesso_mesa_virtual": {
        "tipo": "start",
        "texto": "Acessar Mesa virtual",
        "lane": "Bolsista do RU",
        "proximos": [
          {
            "destino": "cadastrar_doc_sipac"
          }
        ]
      },
      "cadastrar_doc_sipac": {
        "tipo": "tarefa",
        "texto": "Cadastrar documento no SIPAC",
        "lane": "Bolsista do RU",
        "proximos": [
          {
            "destino": "preencher_dados"
          }
        ]
      },
      "preencher_dados": {
        "tipo": "tarefa",
        "texto": "Preencher dados",
        "lane": "Bolsista do RU",
        "proximos": [
          {
            "destino": "escrever_doc"
          }
        ]
      },
      "escrever_doc": {
        "tipo": "tarefa",
        "texto": "Escrever documento",
        "lane": "Bolsista do RU",
        "proximos": [
          {
            "destino": "incluir_assinante"
          }
        ]
      },
      "incluir_assinante": {
        "tipo": "tarefa",
        "texto": "Incluir assinante",
        "lane": "Bolsista do RU",
        "proximos": [
          {
            "destino": "assinar"
          }
        ]
      },
      "assinar": {
        "tipo": "tarefa",
        "texto": "Assinar",
        "lane": "Bolsista do RU",
        "proximos": [
          {
            "destino": "inserir_dados_discente"
          }
        ]
      },
      "inserir_dados_discente": {
        "tipo": "tarefa",
        "texto": "Inserir dados do interessado (discentes)",
        "lane": "Bolsista do RU",
        "proximos": [
          {
            "destino": "salvar_email_discente"
          }
        ]
      },
      "salvar_email_discente": {
        "tipo": "tarefa",
        "texto": "Salvar e-mail cadastrado do(a) discente para envio posterior",
        "lane": "Bolsista do RU",
        "proximos": [
          {
            "destino": "direcionar_doc_unidade_dasap"
          }
        ]
      },
      "direcionar_doc_unidade_dasap": {
        "tipo": "tarefa",
        "texto": "Direcionar documento para unidade DASAP",
        "lane": "Bolsista do RU",
        "proximos": [
          {
            "destino": "receber_doc_dasap"
          }
        ]
      },
      "receber_doc_dasap": {
        "tipo": "tarefa",
        "texto": "DASAP recebe documento",
        "lane": "DASAP",
        "proximos": [
          {
            "destino": "download_declaração_bolsista_ru"
          }
        ]
      },
      "download_declaração_bolsista_ru": {
        "tipo": "tarefa",
        "texto": "Bolsista do RU vai fazer download de declaração",
        "lane": "Bolsista do RU",
        "proximos": [
          {
            "destino": "enviar_via_email"
          }
        ]
      },
      "enviar_via_email": {
        "tipo": "tarefa",
        "texto": "Realizar envio da declaração via e-mail institucional",
        "lane": "Bolsista do RU",
        "proximos": [
          {
            "destino": "finalizar_processo"
          }
        ]
      },
      "finalizar_processo": {
        "tipo": "end",
        "texto": "Finalizar processo",
        "lane": "Bolsista do RU",
        "proximos": []
      }
    }
  }
,
  "boletim_diario.bpmn": {
    "metadata": {
      "nome": "Boletim Diário",
      "id": "1",
      "lanes": [
        "Pauteiro",
        "Repórter",
        "Edição e revisão",
        "Envio e postagem"
      ]
    },
    "elementos": {
      "inicio_multiplo": {
        "tipo": "start",
        "texto": "",
        "lane": "Pauteiro",
        "proximos": [
          {
            "destino": "gateway_viabilidade_pauta",
            "condicao": ""
          }
        ]
      },
      "gateway_viabilidade_pauta": {
        "tipo": "gateway_split",
        "texto": "É viável realizar a pauta?",
        "lane": "Pauteiro",
        "proximos": [
          {
            "destino": "frente_descartar_pauta",
            "condicao": "Não"
          },
          {
            "destino": "estabelecer_agenda_repor",
            "condicao": "Sim"
          }
        ]
      },
      "frente_descartar_pauta": {
        "tipo": "tarefa",
        "texto": "Descartar pauta e encerrar o processo",
        "lane": "Pauteiro",
        "proximos": [
          {
            "destino": "fim_processo",
            "condicao": ""
          }
        ]
      },
      "estabelecer_agenda_repor": {
        "tipo": "tarefa",
        "texto": "Estabelecer agenda de pautas e encaminhar para o repórter",
        "lane": "Pauteiro",
        "proximos": [
          {
            "destino": "apuracao",
            "condicao": ""
          }
        ]
      },
      "apuracao": {
        "tipo": "tarefa",
        "texto": "Apuração",
        "lane": "Repórter",
        "proximos": [
          {
            "destino": "gateway_frente_1",
            "condicao": ""
          }
        ]
      },
      "gateway_frente_1": {
        "tipo": "gateway_split",
        "texto": "",
        "lane": "Repórter",
        "proximos": [
          {
            "destino": "frente_descartar_pauta",
            "condicao": ""
          },
          {
            "destino": "gateway_espera_tempo",
            "condicao": ""
          }
        ]
      },
      "frente_descartar_pauta_2": {
        "tipo": "tarefa",
        "texto": "Descartar pauta e fim",
        "lane": "Repórter",
        "proximos": [
          {
            "destino": "fim_processo",
            "condicao": ""
          }
        ]
      },
      "gateway_espera_tempo": {
        "tipo": "gateway_split",
        "texto": "O tempo de espera é viável?",
        "lane": "Repórter",
        "proximos": [
          {
            "destino": "fim_processo",
            "condicao": "Não"
          },
          {
            "destino": "espera_tempo",
            "condicao": "Sim"
          }
        ]
      },
      "espera_tempo": {
        "tipo": "intermediate_event",
        "texto": "Espera Tempo",
        "lane": "Repórter",
        "proximos": [
          {
            "destino": "frente_sistematizar_matérias",
            "condicao": ""
          }
        ]
      },
      "frente_sistematizar_matérias": {
        "tipo": "tarefa",
        "texto": "Sistematizar as matérias, escrever matéria para a edição",
        "lane": "Edição e revisão",
        "proximos": [
          {
            "destino": "edicao_recebe_matéria",
            "condicao": ""
          }
        ]
      },
      "edicao_recebe_matéria": {
        "tipo": "tarefa",
        "texto": "A edição recebe a matéria",
        "lane": "Edição e revisão",
        "proximos": [
          {
            "destino": "editar_matéria",
            "condicao": ""
          }
        ]
      },
      "editar_matéria": {
        "tipo": "tarefa",
        "texto": "Editar matéria",
        "lane": "Edição e revisão",
        "proximos": [
          {
            "destino": "revisar_orografia",
            "condicao": ""
          }
        ]
      },
      "revisar_orografia": {
        "tipo": "tarefa",
        "texto": "Revisar ortografia",
        "lane": "Edição e revisão",
        "proximos": [
          {
            "destino": "envio_postagem_cadastra_diagramacao",
            "condicao": ""
          }
        ]
      },
      "envio_postagem_cadastra_diagramacao": {
        "tipo": "tarefa",
        "texto": "Envio e postagem cadastra e diagramação",
        "lane": "Envio e postagem",
        "proximos": [
          {
            "destino": "revisar_homologar_boletim",
            "condicao": ""
          }
        ]
      },
      "revisar_homologar_boletim": {
        "tipo": "tarefa",
        "texto": "Revisar e homologar boletim",
        "lane": "Edição e revisão",
        "proximos": [
          {
            "destino": "enviar_email_publico_alvo",
            "condicao": ""
          }
        ]
      },
      "enviar_email_publico_alvo": {
        "tipo": "tarefa",
        "texto": "Enviar por e-mail para o público alvo",
        "lane": "Envio e postagem",
        "proximos": [
          {
            "destino": "cadastrar_portal_redes_sociais",
            "condicao": ""
          }
        ]
      },
      "cadastrar_portal_redes_sociais": {
        "tipo": "tarefa",
        "texto": "Cadastrar no portal e redes sociais",
        "lane": "Envio e postagem",
        "proximos": [
          {
            "destino": "fim_processo",
            "condicao": ""
          }
        ]
      },
      "fim_processo": {
        "tipo": "end",
        "texto": "",
        "lane": "Pauteiro",
        "proximos": []
      }
    }
  },
  "corrida.bpmn": {
    "metadata": {
      "nome": "Corrida",
      "id": "corrida_1",
      "lanes": [
        "Pista"
      ]
    },
    "elementos": {
      "start_corrida": {
        "tipo": "start",
        "texto": "Começar corrida",
        "lane": "Pista",
        "proximos": [
          {
            "destino": "correndo",
            "condicao": ""
          }
        ]
      },
      "correndo": {
        "tipo": "tarefa",
        "texto": "Correndo",
        "lane": "Pista",
        "proximos": [
          {
            "destino": "pulando",
            "condicao": ""
          },
          {
            "destino": "fim",
            "condicao": ""
          }
        ]
      },
      "pulando": {
        "tipo": "tarefa",
        "texto": "Pulando",
        "lane": "Pista",
        "proximos": [
          {
            "destino": "correndo",
            "condicao": ""
          }
        ]
      },
      "fim": {
        "tipo": "end",
        "texto": "Terminar corrida",
        "lane": "Pista",
        "proximos": []
      }
    }
  }


}
from difflib import get_close_matches
def corrigir_destinos(self, limite_similaridade=0.75):
    chaves = list(self['elementos'].keys())
    for id_no, no in self['elementos'].items():
        for prox in no['proximos']:
            if isinstance(prox, dict):
                destino = prox['destino']
                if destino == "":
                    continue
                else:
                    inicio = chaves.index(id_no)
                    chaves_list = chaves[inicio+1:]
                    print(chaves_list)
                    similar = get_close_matches(
                        destino,
                        chaves_list,
                        n=1,
                        cutoff=limite_similaridade
                    )
                    if similar and destino != similar[0]:
                       prox['destino'] = similar[0]
                       print("diferente")
    return  self
# Execução em lote (Batch processing)
if __name__ == "__main__":
    print("[*] Iniciando a compilação automatizada de processos por formulário...")
    for nome_arquivo, payload in formularios_completos.items():
        payload = corrigir_destinos(payload)
        motor = MotorGeradorBPMN(payload)
        #motor.validar()
        motor.salvar(nome_arquivo)
        print(f" [+] Arquivo '{nome_arquivo}' compilado com sucesso.")
    print("[*] Concluído! Todos os diagramas estão prontos para importação.")

Limpando memória

In [ ]:
import torch
import gc

# Limpa a memória cache do PyTorch
torch.cuda.empty_cache()

# Força o coletor de lixo do Python a liberar objetos não utilizados
gc.collect()

Adicionando LORA para fine tuning

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import torch

lora_path = "/content/qwen-bpmn-lora-final"
model2 = PeftModel.from_pretrained(model, lora_path)

PIPE Modelo

In [ ]:
# pipeline
pipe = pipeline(
    "text-generation",
    model=model2,
    tokenizer=tokenizer
)

Executar o Gradio para interagir com o modelo

In [ ]:
import gradio as gr
import json
import re
from datetime import datetime
from transformers import GenerationConfig
import torch
import gc

def corrigir_fechamento(texto):

    pilha = []
    fechamento = {
        "{": "}",
        "[": "]"
    }

    for c in texto:

        if c in "{[":
            pilha.append(c)

        elif c == "}":
            if pilha and pilha[-1] == "{":
                pilha.pop()

        elif c == "]":
            if pilha and pilha[-1] == "[":
                pilha.pop()

    # acrescenta os fechamentos na ordem inversa
    while pilha:
        texto += fechamento[pilha.pop()]

    return texto
def montar_prompt(texto):

    return f"""<s>[INST]
Você é um especialista em BPMN 2.0 e Engenharia de Dados.

Converta o texto fornecido para JSON BPMN válido, seguindo RIGOROSAMENTE as regras abaixo.

RETORNE APENAS O JSON.
NÃO explique.
NÃO escreva texto fora do JSON.
NÃO use Markdown.

REGRAS:

1. Use apenas os tipos permitidos:
   ✓ "start"
   ✓ "tarefa"
   ✓ "subprocesso"
   ✓ "gateway_split"
   ✓ "gateway_merge"
   ✓ "gateway_parallel"
   ✓ "gateway_parallel_merge"
   ✓ "gateway_inclusivo"
   ✓ "gateway_inclusivo_merge"
   ✓ "intermediate_event"
   ✓ "intermediate_event_merge"
   ✓ "end"

2. MAPEAMENTO SEMÂNTICO ESTRITO:
  ✓ Início do processo, gatilho inicial ou evento que inicia o fluxo → "start"
  ✓ Ação, atividade, execução, preenchimento, envio, análise, conferência ou qualquer trabalho realizado por um participante → "tarefa"
  ✓ Decisão, validação, aprovação, verificação, disponibilidade ou qualquer ponto do processo que resulte em dois ou mais caminhos alternativos → "gateway_split"
  ✓ Ponto de reunião de caminhos alternativos originados por um gateway_split (somente um caminho retorna) → "gateway_merge"
  ✓ Abertura de atividades executadas simultaneamente (AND) → "gateway_parallel"
  ✓ Fechamento de atividades paralelas obrigatórias (AND Join), aguardando todos os ramos concluírem → "gateway_parallel_merge"
  ✓ Abertura de fluxos opcionais, onde um ou mais caminhos podem ser executados simultaneamente (OR) → "gateway_inclusivo"
  ✓ Fechamento de fluxos opcionais (OR Join), aguardando apenas os ramos que realmente foram ativados→ "gateway_inclusivo_merge"
  ✓ Evento intermediário de espera, mensagem, tempo, sinal ou sincronização durante o fluxo → "intermediate_event"
  ✓ Evento intermediário utilizado para reunir ou sincronizar fluxos que convergem em um evento → "intermediate_event_merge"
  ✓ Processo composto por um fluxo interno detalhado → "subprocesso"
  ✓ Encerramento do processo ou término de um ramo  → "end"

3. Estrutura obrigatória do JSON:
{{
  "<nome_arquivo>.bpmn": {{
    "metadata": {{
      "nome": "",
      "id": "",
      "lanes": []
    }},
    "elementos": {{
        "": {{
        "tipo":"",
        "texto":"",
        "lane":"",
        "proximos":[{{"destino": "", "condicao": ""}}]
      }}
    }}
  }}
}}

CRITÉRIOS E EXEMPLOS OBRIGATÓRIOS PARA ESCOLHA DE GATEWAYS:
  1. gateway_split (Exclusivo / XOR):
    - QUANDO USAR: Quando ocorrer duas ou mais saídas com condições de escolha. (Ex: "Aprovado ou Reprovado?", "Sim ou Não?").
    - Exemplo:
      "gateway_aprovacao": {{
        "tipo": "gateway_split",
        "texto": "O documento foi aprovado?",
        "lane": "<Nome da Lane>",
        "proximos": [
          {{"destino": "tarefa_seguir", "condicao": "Sim"}},
          {{"destino": "tarefa_revisar", "condicao": "Não"}}
        ]
      }}

  2. gateway_parallel (Paralelo / AND):
    - QUANDO USAR: Quando ocorrer duas ou mais saídas sem condições de escolha.
    - Exemplo:
      "gateway_abrir_paralelo": {{
        "tipo": "gateway_parallel",
        "texto": "",
        "lane": "<Nome da Lane>",
        "proximos": [
          {{"destino": "realizar_inspecao", "condicao": "Frente Manutenção"}},
          {{"destino": "verificar_docs", "condicao": "Frente Documentação"}}
        ]
      }}

  3. gateway_inclusivo (Inclusivo / OR):
    - QUANDO USAR: Quando podem ocorrer um ou mais saídas opcionais e combinados com base em condições.
    - Exemplo:
      "gateway_inclusivo_ajustes": {{
        "tipo": "gateway_inclusivo",
        "texto": "Quais ajustes são necessários?",
        "lane": "<Nome da Lane>",
        "proximos": [
          {{"destino": "revisao_dados", "condicao": "Apontamentos estatísticos"}},
          {{"destino": "revisao_estilo", "condicao": "Apontamentos gramaticais"}}
        ]
      }}

  4. gateway_merge (Convergência):
    - QUANDO USAR: Para unificar caminhos que se dividiram anteriormente por gateway_split.
    - Exemplo:
      "merge_convergencia": {{
        "tipo": "gateway_merge",
        "texto": "",
        "lane": "<Nome da Lane>",
        "proximos": ["reunir_resultados"]
      }}
  5. gateway_parallel_merge (Convergência):
    - QUANDO USAR: Para unificar caminhos que se dividiram anteriormente por gateway_parallel .
    - Exemplo:
      "merge_convergencia": {{
        "tipo": "gateway_parallel_merge",
        "texto": "",
        "lane": "<Nome da Lane>",
        "proximos": ["reunir_resultados"]
      }}
  6. gateway_inclusivo_merge (Convergência):
    - QUANDO USAR: Para unificar caminhos que se dividiram anteriormente por gateway_inclusivo.
    - Exemplo:
      "merge_convergencia": {{
        "tipo": "gateway_inclusivo_merge",
        "texto": "",
        "lane": "<Nome da Lane>",
        "proximos": ["reunir_resultados"]
      }}

VALIDAÇÃO FINAL OBRIGATÓRIA ANTES DE GERAR:
✓ Não existem elementos órfãos (todo nó possui ao menos uma entrada, exceto o start).
✓ Todos os elementos citados em "proximos" existem no dicionário de elementos (sem chaves órfãs ou erradas).
✓ O campo "proximos" de "subprocesso" e "end" é exatamente [].
✓ Todo processo deve possuir pelo menos um elemento do tipo "end".
✓ Apenas gateways podem possuir múltiplos destinos.
✓ Nenhuma tarefa pode possuir dois ou mais elementos em "proximos".
✓ Toda pergunta (texto terminado com "?") deve ser representada por um "gateway_split", nunca por uma "tarefa".
TEXTO:
{texto}
[/INST]
"""

# ============================================================
# LIMPEZA DO JSON
# ============================================================

def extrair_json(texto):

    texto = texto.strip()

    match = re.search(r'\{.*\}', texto, re.DOTALL)

    if match:
        return match.group()

    return texto

# ============================================================
# GERA JSON BPMN
# ============================================================

def gerar_json_bpmn(descricao):
    print("Iniciando")

    prompt = montar_prompt(descricao)


    # =====================================
    # CHAMADA AO MODELO
    # =====================================
    # 1. Configuração focada em velocidade e rigidez estrutural
    gen_config = GenerationConfig(
        max_new_tokens=8048,  # Linha única gasta pouquíssimo token, 2048 sobra muito!
        do_sample=False,
        temperature=0.7,
        pad_token_id=pipe.tokenizer.eos_token_id,
    )

    # 2. Execução da Pipeline
    resp = pipe(prompt, generation_config=gen_config, return_full_text=False)
    resposta_bruta = resp[0]["generated_text"]
    print(resposta_bruta)
    string_limpa = resposta_bruta.strip()
    match = re.search(r"(\{.*\})", string_limpa, re.DOTALL)

    # Só avança se o Regex realmente capturou algo
    if match:
        json_linha_unica = match.group(1)

        try:
            # Converte para dicionário Python
            dados = json.loads(json_linha_unica)
            print("✅ JSON extraído e convertido com sucesso!")
            print(f"Nome do Processo Identificado: {dados}")
            return json.dumps(
                dados,
                ensure_ascii=False,
                indent=2
            )

        # Esse try/except protege seu código caso o JSON venha quebrado/cortado pelo modelo
        except json.JSONDecodeError as e:
            print("Tentativa de Correção")
            texto_corrigido = corrigir_fechamento(json_linha_unica)
            try:
                dados = json.loads(texto_corrigido)
                print("✅ JSON extraído e convertido com sucesso!")
                return json.dumps(
                dados,
                ensure_ascii=False,
                indent=2
            )
            except json.JSONDecodeError as e:
                print("❌ Falha crítica no Parseamento do JSON")
                print(f"Conteúdo bruto capturado: {resposta_bruta}")
                return (
                f"ERRO AO INTERPRETAR JSON\n\n"
                f"{str(e)}\n\n"
                f"RESPOSTA:\n{resposta_bruta}"
            )

    else:
        # Se caiu aqui, é porque a resposta do modelo não tinha chaves {}
        print("❌ Nenhum padrão JSON foi encontrado na resposta.")

# ============================================================
# GERA ARQUIVO JSON
# ============================================================

def salvar_json(json_texto):

    nome = (
        f"bpmn_"
        f"{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    )

    with open(nome, "w", encoding="utf-8") as f:
        f.write(json_texto)

    return nome

# ============================================================
# GERA ARQUIVO BPMN
# ============================================================

def salvar_bpmn(json_texto):
    try:
        json_texto = json.loads(json_texto)
        nome_arquivo = list(json_texto.keys())[0]
        estrutura = json_texto[nome_arquivo]
        estrutura = corrigir_destinos(estrutura)
        motor = MotorGeradorBPMN(estrutura)
        motor.salvar(nome_arquivo)
        return (nome_arquivo)

    except Exception as e:

        raise gr.Error(
            f"Erro ao gerar BPMN: {str(e)}"
        )

def limpar():
    # Limpa a memória cache do PyTorch
    torch.cuda.empty_cache()

    # Força o coletor de lixo do Python a liberar objetos não utilizados
    gc.collect()
    return "", "", None, None
# ============================================================
# INTERFACE
# ============================================================

with gr.Blocks(
    title="Gerador BPMN"
) as demo:

    gr.Markdown("# 🤖 Gerador Automatizado de Grafos BPMN com Robozito")

    descricao = gr.Textbox(
        label="Descrição do Processo",
        lines=20,
        placeholder="""
Exemplo:
Processo de Solicitação de Equipamento
O colaborador registra uma solicitação de equipamento no sistema.
O setor de TI analisa a solicitação para verificar se as informações estão completas.
Caso as informações não estejam completas, o setor de TI solicita correção ao colaborador. Após a correção, a solicitação deve retornar para análise.
Caso as informações estejam completas, o setor de TI verifica se o equipamento está disponível em estoque.
Se o equipamento estiver disponível, o equipamento é separado para entrega.
Se o equipamento não estiver disponível, o setor de compras realiza a aquisição do equipamento.
Após a separação ou aquisição, o gestor deve aprovar a entrega.
Caso o gestor não aprove a entrega, a solicitação é encerrada.
Caso o gestor aprove a entrega, o equipamento é entregue ao colaborador.
Após a entrega ou encerramento da solicitação, o processo é finalizado.
"""
    )


    with gr.Row():
      analisar = gr.Button(
        "Gerar JSON BPMN",
        variant="primary"
      )
      btn_limpar = gr.Button(
          "🗑️ Limpar"
      )
    with gr.Row():
      json_saida = gr.Code(
    label="JSON BPMN",
    language="json",
    lines=25,
    min_width=700,
    visible=True,
    value='{\n  "Gerador": "Gerador"\n}'
)

    with gr.Row():

        btn_json = gr.Button(
            "⚡Baixar JSON",
            variant="primary"
        )

        btn_bpmn = gr.Button(
            "⚡Gerar BPMN",
            variant="primary"
        )

    arquivo_json = gr.File(
        label="Arquivo JSON"
    )

    arquivo_bpmn = gr.File(
        label="Arquivo BPMN"
    )

    analisar.click(
        gerar_json_bpmn,
        inputs=descricao,
        outputs=json_saida
    )

    btn_json.click(
        salvar_json,
        inputs=json_saida,
        outputs=arquivo_json
    )

    btn_bpmn.click(
        salvar_bpmn,
        inputs=json_saida,
        outputs=arquivo_bpmn
    )

    btn_limpar.click(
    fn=limpar,
    inputs=[],
    outputs=[
        descricao,
        json_saida,
        arquivo_json,
        arquivo_bpmn
    ]
)

# ============================================================
# EXECUÇÃO
# ============================================================

demo.launch(
    share=True,
    debug=True
)